# Milestone 4: GRPO + LoRA debug run (Qwen3-1.7B) on a free Kaggle GPU

Trains Qwen3-1.7B with GRPO and a LoRA adapter on the real-filing **train** split, using the strict reward in
`xbrlbench/rl.py`, then evaluates the trained model on the 500-question held-out sample through the same
`xbrlbench run` path as every other model (non-thinking mode, like the `qwen3-1.7b-nothink` baseline from milestone 3).

**Setup (once):**
1. Kaggle → *Create* → *New notebook* → *File* → *Import notebook* → this file.
2. *Settings*: Accelerator **GPU T4 x2**, Internet **on** (phone-verified account).
   With two GPUs, vLLM generates on GPU 1 while training runs on GPU 0; with one GPU they share it (slower).
3. *Add input* → *Upload* a dataset containing **`questions.jsonl`** (the full corpus, `data/filings/v2/questions.jsonl`)
   and **`heldout_500.jsonl`** from the machine that built the corpus. Only the train split is used for training;
   the export refuses held-out questions.
4. *Run all*. A Kaggle session stops after 12 hours; re-running resumes from the last checkpoint
   if `/kaggle/working/m4` was kept (save a version with *Save output*). Download `/kaggle/working/m4.zip` from *Output* at the end.

No API keys are needed and nothing is billed. Start with `MAX_STEPS = 20` to check everything runs.

In [ ]:
BRANCH = "claude/project-thread-aoax1i"   # branch with the M4 tooling (main once its PR merges)
MODEL = "Qwen/Qwen3-1.7B"
MAX_STEPS = 300           # 20 for a smoke run
TRAIN_LIMIT = None        # e.g. 2000 prompts; None = the whole train split
EXCLUDE_FAMILIES = ""     # e.g. "gross_margin,current_ratio" if the base model already solves them
MAX_PROMPT_TOKENS = 3072
MAX_COMPLETION_TOKENS = 768
EVAL_BASE = True          # also evaluate the untrained model here, for a same-machine comparison
EVAL_LIMIT = None         # e.g. 50 for a quick check
WORKERS = 64

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv
rm -rf /kaggle/working/XBRLBench
git clone -q https://github.com/allisonzhan/XBRLBench /kaggle/working/XBRLBench

In [ ]:
import glob
import os
import subprocess

REPO = "/kaggle/working/XBRLBench"
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
# One resolver pass so vLLM and the TRL pins agree on torch/transformers.
subprocess.run(["pip", "install", "-q", "vllm==0.31.0", "-e", f"{REPO}[train]"], check=True)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"Add a dataset containing {name} (see the setup steps above)"
    return hits[0]

QUESTIONS, HELDOUT = find("questions.jsonl"), find("heldout_500.jsonl")
OUT = "/kaggle/working/m4"
TRAIN = f"{OUT}/data/train.jsonl"
cmd = ["python", "-m", "xbrlbench", "export-rl", "--in", QUESTIONS, "--out", TRAIN, "--val-fraction", "0.05"]
if EXCLUDE_FAMILIES:
    cmd += ["--exclude-families", EXCLUDE_FAMILIES]
subprocess.run(cmd, check=True, cwd=REPO)
N_GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
assert N_GPUS >= 1, "No GPU: set Settings -> Accelerator to a GPU"
print("GPUs:", N_GPUS)

In [ ]:
import signal
import time
import urllib.request


def wait_healthy(proc, log, url="http://localhost:8000/health", minutes=15):
    for _ in range(minutes * 12):
        try:
            urllib.request.urlopen(url, timeout=2)
            return proc
        except Exception:
            if proc.poll() is not None:
                raise RuntimeError(open(log).read()[-3000:]) from None
            time.sleep(5)
    raise RuntimeError(f"server did not start in {minutes} minutes; see {log}")


def start(cmd, log, env=None):
    f = open(log, "w")
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
                            env={**os.environ, **(env or {})})
    return wait_healthy(proc, log)


def stop(proc):
    os.killpg(proc.pid, signal.SIGTERM)
    proc.wait(timeout=120)
    time.sleep(10)

## Train

In [ ]:
train_cmd = ["python", f"{REPO}/scripts/train_grpo.py", "--train", TRAIN, "--out", f"{OUT}/run",
             "--model", MODEL, "--max-steps", str(MAX_STEPS), "--merge",
             "--max-prompt-tokens", str(MAX_PROMPT_TOKENS), "--max-completion-tokens", str(MAX_COMPLETION_TOKENS)]
if TRAIN_LIMIT:
    train_cmd += ["--limit", str(TRAIN_LIMIT)]
if glob.glob(f"{OUT}/run/checkpoint-*"):
    train_cmd.append("--resume")

server = None
if N_GPUS >= 2:
    # vLLM generation server on GPU 1; TRL pushes the updated weights to it every step.
    server = start(["trl", "vllm-serve", "--model", MODEL, "--dtype", "half", "--gpu_memory_utilization", "0.9",
                    "--max_model_len", str(MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS)],
                   "/kaggle/working/vllm_train.log", env={"CUDA_VISIBLE_DEVICES": "1"})
    train_cmd += ["--vllm", "server"]
    train_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
else:
    train_cmd += ["--vllm", "colocate"]
    train_env = dict(os.environ)
t0 = time.time()
try:
    subprocess.run(train_cmd, check=True, cwd=REPO, env=train_env)
finally:
    if server:
        stop(server)
print(f"trained in {(time.time() - t0) / 60:.0f} min")

## Evaluate on the held-out sample

In [ ]:
def evaluate(model_path, name):
    proc = start(["vllm", "serve", model_path, "--served-model-name", name, "--dtype", "half",
                  "--tensor-parallel-size", str(N_GPUS), "--max-model-len", "8192",
                  "--gpu-memory-utilization", "0.90", "--port", "8000"],
                 f"/kaggle/working/vllm_{name}.log")
    try:
        cmd = ["python", "-m", "xbrlbench", "run", "--in", HELDOUT, "--api-base", "http://localhost:8000/v1",
               "--models", name, "--reasoning", "off", "--max-tokens", str(MAX_COMPLETION_TOKENS * 2),
               "--workers", str(WORKERS), "--out-dir", f"{OUT}/eval/{name}"]
        if EVAL_LIMIT:
            cmd += ["--limit", str(EVAL_LIMIT)]
        if os.path.exists(f"{OUT}/eval/{name}/responses.jsonl"):
            cmd.append("--resume")
        subprocess.run(cmd, check=True, cwd=REPO)
    finally:
        stop(proc)

evaluate(f"{OUT}/run/merged", "qwen3-1.7b-grpo")
if EVAL_BASE:
    evaluate(MODEL, "qwen3-1.7b-nothink")

In [ ]:
import shutil

with open(f"{OUT}/eval/all_responses.jsonl", "w") as out:
    for path in sorted(glob.glob(f"{OUT}/eval/*/responses.jsonl")):
        out.write(open(path).read())
subprocess.run(["python", "-m", "xbrlbench", "report", "--in", f"{OUT}/eval/all_responses.jsonl",
                "--questions", HELDOUT, "--out-prefix", f"{OUT}/eval/report"], check=True, cwd=REPO)
subprocess.run(["python", "-m", "xbrlbench", "reward", "--in", f"{OUT}/eval/all_responses.jsonl"], check=True, cwd=REPO)

# The merged model, the fp16 base copy and checkpoints are large and can be rebuilt; zip the rest.
stage = "/kaggle/working/m4_out"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(OUT, stage, ignore=shutil.ignore_patterns("merged", "base_fp16", "checkpoint-*"))
shutil.make_archive("/kaggle/working/m4", "zip", stage)
print("Download /kaggle/working/m4.zip from the Output tab")